# Root Cause of Scoring Bias  Kaggle Notebook
**Free GPU (P100/T4) · 30 hrs/week · ~$0 cost**

This notebook loads base and instruct models from HuggingFace and compares their scoring bias.

**Models tested:** Llama 3 8B, Mistral 7B, Gemma 2 2B (base + instruct = 6 models)

**Estimated runtime:** 4-6 hours on P100

In [ ]:
# Cell 1: Install dependencies (first run only)
!pip install -q transformers torch accelerate huggingface_hub pandas numpy matplotlib seaborn

import torch, gc, os, csv, json, math, re, time
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Model registry
# Only includes models that work on a single P100 (16GB VRAM)
# HuggingFace login required for Llama and Gemma

from huggingface_hub import login
import getpass

if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HuggingFace token (write permission): ')
    os.environ['HF_TOKEN'] = token
login(token=os.environ['HF_TOKEN'])

MODELS = {
    "llama3-8b": {
        "base": "meta-llama/Meta-Llama-3-8B",
        "instruct": "meta-llama/Meta-Llama-3-8B-Instruct",
    },
    "mistral-7b": {
        "base": "mistralai/Mistral-7B-v0.3",
        "instruct": "mistralai/Mistral-7B-Instruct-v0.3",
    },
    "gemma2-2b": {
        "base": "google/gemma-2-2b",
        "instruct": "google/gemma-2-2b-it",
    },
}
print(f"Registered {len(MODELS)} model families")

In [ ]:
# Cell 3: Load model and run scoring
def load_and_score(model_key, variant, items):
    """Load a model and score evaluation items."""
    hf_id = MODELS[model_key][variant]
    name = f"{model_key}-{variant}"
    
    print(f"Loading {name} ({hf_id})...", end=" ", flush=True)
    t0 = time.time()
    
    tokenizer = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        hf_id,
        torch_dtype=torch.float16,
        device_map="cuda",
        trust_remote_code=True,
    )
    model.eval()
    print(f"OK ({time.time()-t0:.0f}s, {sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")
    
    results = []
    for i, item in enumerate(items):
        prompt = f"""Score the following response from 1 to 5.
### Instruction:\n{item['instruction']}\n
### Response:\n{item['response']}\n
Score:"""
        
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=5,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        match = re.search(r'\b([1-5])\b', response)
        score = int(match.group(1)) if match else None
        results.append(score)
        
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(items)} scored")
    
    # Free memory
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return results

In [ ]:
# Cell 4: Run the experiment
# This tests rubric order bias on base vs instruct models

test_items = [
    {"instruction": "Write a short poem about AI.", "response": "Silicon dreams in digital sleep, neural networks secrets deep."},
    {"instruction": "Explain how databases work.", "response": "A database stores information in tables, like a digital filing cabinet."},
    {"instruction": "What is machine learning?", "response": "Machine learning is a subset of AI where systems learn from data."},
    {"instruction": "Describe the water cycle.", "response": "Water evaporates from oceans, forms clouds, and returns as rain."},
    {"instruction": "Write a sorting function.", "response": "def sort_list(lst): return sorted(lst)"},
]

all_results = {}
for family in MODELS:
    all_results[family] = {}
    for variant in ["base", "instruct"]:
        try:
            scores = load_and_score(family, variant, test_items)
            all_results[family][variant] = scores
        except Exception as e:
            print(f"FAILED: {family}-{variant}: {e}")
            all_results[family][variant] = None
        torch.cuda.empty_cache()

In [ ]:
# Cell 5: Analyze results
print("\n" + "="*60)
print("BASE vs INSTRUCT COMPARISON")
print("="*60)
print(f"{\'Family\':<15} {\'Base Score\':<15} {\'Instruct Score\':<15} {\'Delta\':<10}")
print("-"*55)
for family in all_results:
    base_scores = all_results[family].get("base")
    inst_scores = all_results[family].get("instruct")
    if base_scores and inst_scores:
        bm = sum(s for s in base_scores if s) / len([s for s in base_scores if s])
        im = sum(s for s in inst_scores if s) / len([s for s in inst_scores if s])
        delta = im - bm
        print(f"{family:<15} {bm:<15.3f} {im:<15.3f} {delta:<+.3f}")
    else:
        print(f"{family:<15} FAILED")

In [ ]:
# Cell 6: Save results
import json
save_path = "/kaggle/working/rootcause_results.json"
with open(save_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Results saved: {save_path}")
print(f"\nTo download: Kaggle sidebar → Output → Data → rootcause_results.json")

## Instructions
1. **HuggingFace token**: Get from huggingface.co/settings/tokens (write permission)
2. **Model access**: Request access for Llama 3 and Gemma 2 at their HF pages
3. **Kaggle**: New notebook → GPU Accelerator → P100 (30 hrs/week free)
4. **Run**: Cells 1→6 sequentially. Each model takes ~20-45 min to load + score.
5. **Total**: ~4-6 hours for all 6 models on P100.

**Cost: $0 (Kaggle free tier)**